# Feat Importance

In [ ]:
import os
import sys

sys.path.append(os.path.abspath("../"))
from src.config import BASE_PATH, SEED
from src.data_utils import get_data, get_models

import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap, TwoSlopeNorm
from MLstatkit import Delong_test

print(f"Path: {BASE_PATH}")

In [ ]:
# Data
DATA_DICT = {
    "base": get_data(is_nomo=False),
    "nomo": get_data(is_nomo=True),
}


# Models
model_dir = BASE_PATH / "artifacts/models/trained"
model_prefix_list = ["lgbm", "xgb", "knn", "svc", "nn", "stack"]
## Base models
model_dict = get_models(model_prefix_list, model_dir)
## Nomogram
model_dict.update(get_models(["lr"], model_dir))

## Run DeLongs Test for AUROC comparison

In [ ]:
models = list(model_dict.keys())
pval_df = pd.DataFrame(0.0, index=models, columns=models)
winner_df = pd.DataFrame("0", index=models, columns=models)
for model_nameA, modelA in model_dict.items():
    for model_nameB, modelB in model_dict.items():
        if model_nameA == model_nameB:
            continue
        if model_nameA == "lr":
            X_test_A = DATA_DICT["nomo"]["X"]
        else:
            X_test_A = DATA_DICT["base"]["X"]
        if model_nameB == "lr":
            X_test_B = DATA_DICT["nomo"]["X"]
        else:
            X_test_B = DATA_DICT["base"]["X"]
        y_test = DATA_DICT["base"]["y"].values.ravel()
        y_proba_A = modelA.predict_proba(X_test_A)[:, 1]
        y_proba_B = modelB.predict_proba(X_test_B)[:, 1]
        results = Delong_test(
            true=y_test,
            prob_A=y_proba_A,
            prob_B=y_proba_B,
            alpha=0.95,
            return_auc=True,
            return_ci=True,
            n_boot=1,
            random_state=SEED,
            verbose=0,
        )
        p_val = results[1]
        # p_val = format_p_val(p_val)
        auc_A = results[4]
        auc_B = results[5]
        if auc_A > auc_B:
            best_model = "A"  # row is better
        else:
            best_model = "B"  # col is better
        pval_df.loc[model_nameA, model_nameB] = p_val

        winner_df.loc[model_nameA, model_nameB] = best_model

mask = np.zeros_like(pval_df)
mask[np.triu_indices_from(mask)] = True
fig, ax = plt.subplots(figsize=(10, 8))
colors = [(0.0, "green"), (0.15, "purple"), (1.0, "blue")]  # 0  # 0.05  # 1
custom_cmap = LinearSegmentedColormap.from_list("custom_pval", colors)
norm = TwoSlopeNorm(vmin=0, vcenter=0.05, vmax=1)
sns.heatmap(
    pval_df,
    cmap=custom_cmap,
    annot=True,
    fmt=".3f",
    # center=0.05,
    vmin=0,
    vmax=1,
    # center=1,
    linewidths=1,
    # linecolor="grey",
    mask=mask,
    square=True,
    cbar_kws={"label": "p-value"},
    ax=ax,
)
for i in range(pval_df.shape[0]):
    for j in range(pval_df.shape[1]):
        if i <= j or pval_df.iloc[i, j] > 0.05:  # type: ignore
            continue
        winner = winner_df.iloc[i, j]
        # Place the marker: could use text or Unicode triangle/arrow
        if winner == "A":  # Row model wins
            annotation = "\u2190"  # Left triangle (row wins)
        else:
            annotation = "\u2193"  # Down triangle (column wins)
        ax.text(
            j + 0.5,
            i + 0.75,
            annotation,
            va="center",
            ha="center",
            fontsize=12,
            color="black",
        )

plt.xticks(rotation=45, ha="right")
plt.yticks(rotation=0)
plt.title(f"ORN- Model vs Model AUROC comparison")
plt.tight_layout()
save_path = BASE_PATH / "results/figures/sig.pdf"
if save_path.exists():
    save_path.unlink()
save_path.parent.mkdir(exist_ok=True, parents=True)
plt.savefig(save_path, bbox_inches="tight")